In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

from typing import Any

In [19]:
class GPTDatasetV1(Dataset):
  def __init__(self, txt: str, tokenizer: Any, max_length: int, stride: int) -> None:
    self.input_ids: list[torch.Tensor] = []
    self.target_ids: list[torch.Tensor] = []
    # テキスト全体をtoken化
    token_ids = tokenizer.encode(txt)
    self.token_size = len(token_ids)

    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i : i + max_length]
      offset = 1
      target_chunk = token_ids[i + offset : i + max_length + offset]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self) -> int:
    return len(self.input_ids)

  def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
    return self.input_ids[index], self.target_ids[index]

In [37]:
def create_dataloader_v1(
  txt: str,
  batch_size: int = 4,
  max_length: int = 256,
  stride: int = 128,
  shuffle: bool = True,
  drop_last: bool = True,
  num_workers: int = 0,
) -> Any:
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
  return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

In [39]:
with open("the-verdict.txt", encoding="utf-8") as f:
  raw_text = f.read()

# これはgpt2の語彙数
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"{inputs=}")
token_embeddings = token_embedding_layer(inputs)
print(f"{token_embeddings.shape}")

inputs=tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
torch.Size([8, 4, 256])


In [40]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [42]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
